In [ ]:
"""from datasets import load_dataset

# Loading 50,000 stories
dataset = load_dataset("roneneldan/TinyStories", split="train[:50000]")

# Saving everything as a text file
with open("data/tinystories_extract.txt", "w", encoding="utf-8") as f:
    for story in dataset:
        f.write(story['text'] + "\n<|endoftext|>\n")"""

# Only needs to be ran once

In [18]:
"""from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel

tokenizer = Tokenizer(BPE())
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)

vocab_size = 2000

trainer = BpeTrainer(
    vocab_size=vocab_size,
    special_tokens=["<|endoftext|>"],
    show_progress=True
)

file = ["data/tinystories_extract.txt"]

tokenizer.train(file, trainer=trainer)

tokenizer.save(f"weights/tiny_stories_50ktrain_{vocab_size}.json")
print("BPE training completed. Vocab size is :", tokenizer.get_vocab_size())"""
# Only needs to be ran once too

'from tokenizers import Tokenizer\nfrom tokenizers.models import BPE\nfrom tokenizers.trainers import BpeTrainer\nfrom tokenizers.pre_tokenizers import ByteLevel\n\ntokenizer = Tokenizer(BPE())\ntokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)\n\nvocab_size = 2000\n\ntrainer = BpeTrainer(\n    vocab_size=vocab_size,\n    special_tokens=["<|endoftext|>"],\n    show_progress=True\n)\n\nfile = ["data/tinystories_extract.txt"]\n\ntokenizer.train(file, trainer=trainer)\n\ntokenizer.save(f"weights/tiny_stories_50ktrain_{vocab_size}.json")\nprint("BPE training completed. Vocab size is :", tokenizer.get_vocab_size())'

In [3]:
import numpy as np
from src.ai_lib import layers
from src.ai_lib import Model, metrics, SequenceDataLoader, CosineWarmupScheduler
from src.ai_lib import optimizers
from src.ai_lib import losses
from src.ai_lib import preprocessing

vocab_size = 2000

tokenizer = preprocessing.CustomBPETokenizer(path= f"weights/tiny_stories_50ktrain_{vocab_size}.json")

with open("data/tinystories_extract.txt", "r", encoding="utf-8") as f:
    text_data = f.read()

import numpy as np
data_tensor = np.array(tokenizer.encode(text_data), dtype=np.int32)

print(tokenizer.vocab_size)

2000


In [4]:
train_test_split = 0.9

n = int(train_test_split * len(data_tensor))
train_data = data_tensor[:n]
val_data = data_tensor[n:]

batch_size = 128
block_size = 100  # Numbers of tokens to look back at
steps_per_epoch = 25 

# Creating dataloaders
train_loader = SequenceDataLoader(
    data=train_data, 
    block_size=block_size, 
    batch_size=batch_size, 
    steps_per_epoch=steps_per_epoch
)

val_loader = SequenceDataLoader(
    data=val_data, 
    block_size=block_size, 
    batch_size=batch_size, 
    steps_per_epoch=5 # Less steps for validation
)

In [5]:
from functools import partial
d_model = 256
n_heads = 8
d_ff = 682
n_kv_heads = 4

dropout_rate = 0.1

vocab_size = tokenizer.vocab_size

norm_builder = partial(layers.RMSNorm, n_features=d_model)

attn_builder = partial(layers.GQARoPEAttentionHead, n_heads=n_heads, n_kv_heads = n_kv_heads, d_model=d_model, is_causal=True, block_size=block_size)

ffn_builder = partial(layers.SwiGLU, in_features=d_model, hidden_features=d_ff)

embedding_layer = layers.Embedding(vocab_size=vocab_size, d_model=d_model)

llm_network = layers.Sequential([
    embedding_layer,

    # 6 Transformers
    layers.TransformerBlock(attention_builder=attn_builder, norm_builder=norm_builder,ffn_builder=ffn_builder, dropout_rate=dropout_rate),
    layers.TransformerBlock(attention_builder=attn_builder, norm_builder=norm_builder,ffn_builder=ffn_builder, dropout_rate=dropout_rate),
    layers.TransformerBlock(attention_builder=attn_builder, norm_builder=norm_builder,ffn_builder=ffn_builder, dropout_rate=dropout_rate),
    layers.TransformerBlock(attention_builder=attn_builder, norm_builder=norm_builder,ffn_builder=ffn_builder, dropout_rate=dropout_rate),
    layers.TransformerBlock(attention_builder=attn_builder, norm_builder=norm_builder,ffn_builder=ffn_builder, dropout_rate=dropout_rate),
    layers.TransformerBlock(attention_builder=attn_builder, norm_builder=norm_builder,ffn_builder=ffn_builder, dropout_rate=dropout_rate),

    layers.RMSNorm(n_features=d_model),

    layers.TiedOutputProjection(embedding_layer)
])

model = Model(llm_network)

In [6]:
model.load_weights("weights/microgpt_v3.npz")

Weights loaded : weights/microgpt_v3.npz


In [37]:
learning_rate = 1e-3
weight_decay = 1e-1

optimizer = optimizers.Adam(learning_rate=learning_rate, weight_decay=weight_decay)
loss = losses.SoftmaxCrossEntropy()

lr_scheduler = CosineWarmupScheduler(lr_max = 3e-3, lr_min = 3e-4, warmup_steps=1600, total_steps=32000)

In [43]:
print("Beginning of training of microgpt_v3")
model.fit(
    dataloader=train_loader,
    epochs=12,
    loss=loss,
    optimizer=optimizer,
    validation_dataloader=val_loader,
    verbose=True,
    lr_scheduler=lr_scheduler,
    metrics=[metrics.accuracy]
)


Beginning of training of microgpt_v3

--- Epoch : 0 ---
Training loss : 2.77167
  accuracy (train) : 0.41223
Validation loss : 2.75948
  accuracy (val) : 0.41669

--- Epoch : 1 ---
Training loss : 2.75807
  accuracy (train) : 0.41576
Validation loss : 2.71705
  accuracy (val) : 0.42024

--- Epoch : 2 ---
Training loss : 2.72491
  accuracy (train) : 0.42122
Validation loss : 2.72182
  accuracy (val) : 0.42292

--- Epoch : 3 ---
Training loss : 2.70476
  accuracy (train) : 0.42523
Validation loss : 2.70805
  accuracy (val) : 0.42298

--- Epoch : 4 ---
Training loss : 2.69803
  accuracy (train) : 0.42560
Validation loss : 2.67927
  accuracy (val) : 0.42595

--- Epoch : 5 ---
Training loss : 2.68156
  accuracy (train) : 0.42692
Validation loss : 2.68899
  accuracy (val) : 0.43399

--- Epoch : 6 ---
Training loss : 2.67585
  accuracy (train) : 0.42901
Validation loss : 2.68788
  accuracy (val) : 0.43422

--- Epoch : 7 ---
Training loss : 2.66408
  accuracy (train) : 0.43056
Validation loss 

In [19]:
def generate_text(model, tokenizer, prompt_text="A ", max_new_tokens=100):
    """
    Generates text
    """
    # Important : Turns off setting
    model.sequential.set_training(False)
    
    # Use of kv_cache for inference
    model.sequential.set_use_cache(True)
    model.sequential.reset_cache()
    
    # Encodes context (initial text / prompt)
    context_ids = tokenizer.encode(prompt_text)
    
    # Prefill
    X_input = np.array([context_ids], dtype=np.int32)
    logits = model.predict(X_input)
    
    # We take the predicted char and add it to the list
    next_char_id = sample_top_k(logits[0, -1, :], temperature=1, top_k=3)
    context_ids.append(next_char_id)
    
    # Decode
    for _ in range(max_new_tokens - 1):
        # X_input becomes a simple integer as we use kv_cache
        X_input = np.array([[next_char_id]], dtype=np.int32)
        
        logits = model.predict(X_input)
        
        # Since X_input is of length one, logits has shape (1, 1, Vocab_size)
        next_char_id = sample_top_k(logits[0, -1, :], temperature=1, top_k=3)
        context_ids.append(next_char_id)
        
    return tokenizer.decode(context_ids)


def sample_top_k(logits, temperature=1, top_k=5):
    """
    Sampling from only the k most likely tokens with temperature
    """

    logits = logits / temperature
    
    indices_to_remove = np.argsort(logits)[:-top_k]
    
    logits[indices_to_remove] = -np.inf
    
    exp_preds = np.exp(logits - np.max(logits))
    preds = exp_preds / np.sum(exp_preds)
    
    return np.random.choice(len(logits), p=preds)

generated_text = generate_text(model, tokenizer, prompt_text="Once upon a time", max_new_tokens=300)
print(generated_text)

Once upon a time, there was a girl named Lily. One day, she went to the park with her mommy and daddy. They found a shiny dress and a shiny coat. Lily was so excited to see what was inside the world.

"Look, Lily! I'm so proud of you, Lily!" she said, holding up the door.

"Thank you, Lily!" Lily said, holding her doll and they both ran to the store and ran to the store.

"Thank you, Mom," Lily said. "You're welcome, mom. I am proud of you. I'm glad we were here."

Lily was happy and hugged her mom and said. "You're welcome, mom. I'm glad you like the toy and I'm glad you like it. I will help you."

Lily was playing in the park. She saw a big, shiny, blue ball. She wanted to see the dog and the dog. She asked the dog to help her.

"Can I have some food?" her mom asked.

"Yes, I can play with the ball," her mom replied.

"No, it's mine!" Lily said.

"Yes, we can. You can't find a toy!" Lily said. She looked up and saw that Lily was not a toy, but she was very sad.

"Mom, what is this?" 

In [45]:
model.save_weights("weights/microgpt_v3.npz")

Model saved : weights/microgpt_v3.npz
